# Chapter 9: The Trouble with Distributed Systems
*From: Designing Data-Intensive Applications*

---

## TL;DR

- A distributed system is one in which a **partial failure** is the norm: some nodes or some network links are broken while others keep working, and no node has a global view of what is still alive.
- The network is **asynchronous and unreliable** — packets can be lost, delayed, duplicated, or reordered — and a node that does not reply is indistinguishable from one that crashed, is paused, or whose reply was lost. Timeouts are the only practical fault detector, and any timeout is a guess.
- Every machine's **clock is a lie** to some degree: quartz drift, leap seconds, NTP jumps, and VM suspension all make wall-clock timestamps unsafe for ordering; a clock reading is really a confidence interval, not a point.
- Any thread can be **paused for seconds** (GC, live-migration, paging, SIGSTOP) at any moment, so a leader that held a valid lease a moment ago may no longer hold it when it finally executes the next line of code.
- Defences are built out of a small toolkit: **majority quorums**, **leases with fencing tokens**, **checksums** for honest-lie (Byzantine-lite) defence, and formal **system models** (synchronous / partially synchronous / asynchronous; crash-stop / crash-recovery / Byzantine) that split correctness into **safety** (nothing bad happens) and **liveness** (something good eventually happens).
- Correctness at this scale needs more than code review: **TLA+ model checking**, **Jepsen-style fault injection**, and **deterministic simulation testing** (FoundationDB, TigerBeetle, Antithesis) turn the astronomical failure space into something you can actually explore.

---

## What distributed-systems software must tolerate

Unlike a single machine — where a fault is usually total and the program simply crashes — a distributed system has to keep working in the presence of every combination of the following:

- **Packets silently dropped or arbitrarily delayed** by switches, routers, NICs, or a neighbour's traffic storm.
- **Asymmetric partitions** where A can reach B but B cannot reach A, and partial partitions where the cluster splits into two groups that each think the other is dead.
- **Timeouts** that could mean "the request was lost," "the node crashed," "the node is alive but paused," "the reply was lost," or "everything is fine, just slow."
- **Clock drift** (Google plans for up to 200 ppm), NTP jumps, slewing, leap seconds, and virtualised clocks that jump forward when the VM is resumed.
- **Unbounded process pauses**: stop-the-world GC, VM live-migration, disk I/O, page faults / swap, SIGSTOP from a debugger, steal time on a noisy neighbour.
- **Zombie leaders**: a node that thinks it is still the leader because it hasn't noticed its lease expired during a pause.
- **Byzantine-lite "honest lies"**: flipped bits from failing RAM, garbled TCP payloads, misconfigured NTP servers reporting a time that is hours off, buggy retry storms.
- **Full Byzantine faults** in adversarial settings (aerospace, blockchains, peer-to-peer): nodes that actively send contradictory messages to different peers.
- **Operator error, misconfigured firewalls, expired TLS certificates, and correlated hardware failure** (same rack, same power phase, same software bug).

The rest of this chapter catalogues the specific hazards and the minimum toolkit used to defend against them.

---

## Back-of-envelope: how much can a clock drift?

Google assumes its servers' quartz clocks may drift by up to **200 parts per million (ppm)** — that is, up to 200 microseconds per second of error relative to true time. The chapter gives two concrete reference points that fall out of this number: ~6 ms after 30 s without resync, and ~17 s after a full day without resync. Let us reproduce those and then ask a slightly different question — how big is the uncertainty interval for an NTP-synchronised clock when the round-trip over a congested internet link spikes?

In [ ]:
# --- Quartz drift at 200 ppm ---
drift_ppm = 200                  # parts per million (Google's assumed worst case)
drift_fraction = drift_ppm / 1_000_000  # = 2e-4 seconds of error per second of time

for window_label, window_seconds in [
    ("30 seconds", 30),
    ("1 minute",   60),
    ("1 hour",     3600),
    ("1 day",      86_400),
    ("1 week",     7 * 86_400),
]:
    drift_seconds = window_seconds * drift_fraction
    print(f"After {window_label:>10}: worst-case drift = {drift_seconds*1000:>12,.2f} ms")

print()

# --- NTP confidence interval over a congested internet link ---
# Uncertainty ~ (quartz drift since last sync) + NTP server uncertainty + RTT/2
sync_interval_s = 60                  # NTP polls once a minute
drift_since_sync_ms = sync_interval_s * drift_fraction * 1000
ntp_server_uncert_ms = 1              # a good stratum-1 server
typical_rtt_ms = 20
spike_rtt_ms   = 400                  # a congested-internet spike from the chapter

typical_ci_ms = drift_since_sync_ms + ntp_server_uncert_ms + typical_rtt_ms/2
spike_ci_ms   = drift_since_sync_ms + ntp_server_uncert_ms + spike_rtt_ms/2

print(f"NTP confidence half-width (typical RTT {typical_rtt_ms} ms): +/- {typical_ci_ms:.2f} ms")
print(f"NTP confidence half-width (spiked  RTT {spike_rtt_ms} ms):   +/- {spike_ci_ms:.2f} ms")
print()
print("Interpretation: a timestamp read with microsecond resolution is still only")
print("accurate to +/- tens of milliseconds on a public-internet-synced node.")
print("Using it to decide 'which write happened first' between datacenters is")
print("unsafe unless you wait out the interval (Spanner) or use logical clocks.")

If you run the cell you get ~6 ms for 30 s and ~17.28 s for a day — matching the book. The takeaway: microsecond resolution is not microsecond accuracy, and an NTP-sync'd clock on the public internet is routinely off by tens of ms even in the best case.

---

## Mental model of failure modes

A single request from a client to a remote service travels through many layers, each of which can fail in its own way. Keep this picture in mind for the rest of the chapter — every deep-dive section is about one of these failure points.

```mermaid
graph LR
  C[Client] -->|1. send| NQ[Sender NIC + kernel queue]
  NQ -->|2. route| NET[Network: switches, routers, public internet]
  NET -->|3. deliver| RK[Remote kernel + recv queue]
  RK -->|4. dispatch| APP[Remote app thread]
  APP -->|5. reply| RNET[Reply network path]
  RNET -->|6. ack| C
  classDef fail fill:#fee,stroke:#c33
  class NQ,NET,RK,APP,RNET fail
```

Failure modes at each hop:

| Hop | What can go wrong |
|---|---|
| **1. Sender queue** | Kernel queue full, connection refused, TCP RTO; SIGSTOP on the sender. |
| **2. Network** | Packet loss, reordering, asymmetric partition, BGP flap, NIC firmware bug. |
| **3. Remote kernel** | Receive queue overflow, SYN flood, NIC dropped frames. |
| **4. Remote app** | App crashed, app paused (GC / live-migration), app leader-lease expired, app lying (Byzantine). |
| **5. Reply path** | Same as 2, plus: response generated but reply socket closed. |
| **6. Client receive** | Client timed out and already retried; client GC'd and missed the reply. |

A timeout at the client cannot tell you **which** of these happened — and that ambiguity is the root of almost every distributed-systems hazard below.

---

## Deep Dive 1 — Unreliable networks and fault detection

Most distributed systems run on **asynchronous packet-switched networks** (Ethernet, IP). Those networks were designed for high utilisation, not for bounded delay: a queue at a switch can grow without limit, TCP can retransmit indefinitely, and the only "sorry, can't deliver" signal you usually get is silence. Circuit-switched networks (phone lines) offer bounded latency because bandwidth is reserved per call; packet networks deliberately trade that guarantee for statistical multiplexing. Datacenter fabrics lie somewhere in the middle, but even there a microburst or a failing switch can produce seconds of queuing.

The practical consequence is that **no node can distinguish** a lost request, a queued-but-not-yet-delivered request, a crashed receiver, a paused receiver, a lost reply, or a reply that is still on its way. All six look identical from the sender's point of view. The only tool available is a **timeout**, and every choice of timeout is a guess: too short and you declare healthy nodes dead (false positive → cascading failure, needless failover, duplicate work); too long and the system freezes while users wait.

```mermaid
graph TD
  Send[Client sends request] --> Q{No reply after T seconds}
  Q --> L1[lost on the wire]
  Q --> L2[queued at a switch]
  Q --> L3[remote node crashed]
  Q --> L4[remote node paused]
  Q --> L5[reply was lost]
  Q --> L6[reply is still in flight]
```

Practical fault detection therefore uses **adaptive timeouts** driven by observed RTT distributions, **phi-accrual-style suspicion** (Akka, Cassandra), and **quorum confirmation** — a single node never decides another is dead on its own; the cluster agrees.

See [[01-unreliable-networks-and-fault-detection]].

---

## Deep Dive 2 — Unreliable clocks and TrueTime

Every machine has **two different clocks** and they answer different questions. A **monotonic clock** (`CLOCK_MONOTONIC`, `System.nanoTime`) only moves forward and is perfect for measuring elapsed time on one machine; its absolute value is meaningless across machines and it cannot be compared to another node's clock. A **time-of-day clock** (`CLOCK_REALTIME`, `System.currentTimeMillis`) tracks wall-clock time, is usually kept in line by NTP, and can be compared across machines — but it can **jump backwards**, **leap-skip**, get **slewed**, be off by hours when NTP is misconfigured, and pause-jump inside a VM.

Because of this, a single timestamp reading is better thought of as a **confidence interval**: "the real time is somewhere in `[earliest, latest]`." Most OS APIs hide this interval — `clock_gettime` returns nanoseconds but gives no error bound — but Google's **TrueTime** and Amazon's **ClockBound** expose it explicitly. Spanner then uses the interval to serialise global transactions: after committing, it **waits out the uncertainty window** so that any later transaction's earliest-possible timestamp is strictly after this one's latest-possible timestamp.

```mermaid
sequenceDiagram
  participant T as TrueTime API
  participant S as Spanner leader
  participant L as Later reader
  S->>T: TT.now()
  T-->>S: [earliest=100, latest=107]
  Note over S: pick commit_ts = latest = 107
  S->>S: commit wait (sleep 7 ms)
  Note over S: now TT.now().earliest > 107
  L->>T: TT.now()
  T-->>L: [earliest=115, latest=122]
  Note over L: reads see commit at 107 as<br/>strictly in the past
```

The corollary: using a wall-clock timestamp for **last-write-wins** (as early Cassandra and ScyllaDB did) means a client with a skewed clock can silently overwrite newer data — a bug that leaves no trace. Logical clocks (Lamport, vector, hybrid logical clocks) are the safer default when you only need ordering, not real time.

See [[02-unreliable-clocks]].

---

## Deep Dive 3 — Process pauses and arbitrary preemption

A thread running on a commodity server can be **stopped for seconds or minutes** at any instruction boundary, and then resumed as if nothing happened. Sources of pauses include: **stop-the-world GC** (a single HBase incident has been observed at multiple minutes), **VM live-migration** (tens of seconds while memory is copied), **kernel context switches and steal time** on over-subscribed hypervisors, **page faults / swap thrashing**, **slow synchronous disk I/O** from an unrelated library call, and literal `SIGSTOP` from an operator's debugger.

This matters because code like *"check the lease, then act on it"* is not safe on such a machine:

```mermaid
sequenceDiagram
  participant L as Leader
  participant Lock as Lock service
  participant S as Storage
  L->>Lock: renew lease, expires at t=30s
  Note over L: t=25s: check isValid() -> yes
  Note over L: pause 15 seconds (GC / VM suspend)
  Note over L: t=40s: resume and call process()
  L->>S: write "I am the leader"
  Note over S: but the lease expired at t=30s<br/>a new leader may already exist
```

Defences against this are structural, not code-level: **fencing tokens** (next section), **bounded real-time systems** (aerospace, automotive — expensive, specialised), or simply **assume pauses will happen and make every write idempotent and checked at the storage layer**.

See [[03-process-pauses]].

---

## Deep Dive 4 — Quorums, leases, and fencing tokens

Because no single node can trust its own view of who is alive, distributed algorithms decide by **majority quorum**: a decision is valid only if *more than half* the nodes agree. This tolerates *f* failures in a cluster of `2f+1` nodes and guarantees that two disjoint quorums must overlap in at least one node (so they cannot both make contradictory decisions).

A **lease** is a time-limited lock: one node is allowed to act as leader for a bounded interval, after which it must renew or give up. Leases cut the coordination cost relative to asking the quorum on every request — but they are fragile under process pauses. The fix is **fencing tokens**: the lock service hands out a **monotonically increasing token** with each lease, and the **storage layer** rejects any write whose token is lower than the highest one it has already seen.

```mermaid
sequenceDiagram
  participant C1 as Client 1
  participant C2 as Client 2
  participant LS as Lock service
  participant S as Storage
  C1->>LS: acquire lease
  LS-->>C1: token = 33
  Note over C1: pause 20s (GC)
  C2->>LS: acquire lease (C1's lease expired)
  LS-->>C2: token = 34
  C2->>S: write, token=34
  S-->>C2: ok (34 >= last=34)
  C1->>S: write, token=33 (zombie!)
  S-->>C1: REJECT (33 < last=34)
```

The critical property is that the **storage layer** itself checks the token. This protects against *both* zombie ex-leaders (their tokens are stale) and delayed packets (same reason). Alternatives like **STONITH** ("shoot the other node in the head") can evict a zombie, but they don't save you from a delayed packet that was sent before the eviction and is still in a switch queue.

Conditional writes on storage (S3 preconditions, etcd revision numbers, DynamoDB `ConditionExpression`) are a popular way to get fencing without running a separate lock service.

See [[04-quorums-leases-and-fencing-tokens]].

---

## Deep Dive 5 — Byzantine faults

A **Byzantine fault** is when a node *actively lies* — sending different messages to different peers, forging signatures, claiming to have committed data it discarded. This is the strongest failure model and the most expensive to defend against: full **Byzantine fault tolerant (BFT)** consensus requires roughly **2/3 honest nodes** (`3f+1` total to tolerate `f` liars), multiple independent implementations, and complex protocols (PBFT, HotStuff). BFT is economically justified in **aerospace and defence** (a cosmic ray can flip bits), **cryptocurrencies and public blockchains** (participants are mutually distrusting), and some **peer-to-peer** systems — but not inside a single organisation's datacenter, where firewalls, authentication, and code review are cheaper defences.

What *is* worth defending against in ordinary systems is **"honest lying"** — corrupted or malformed messages sent by buggy or failing (not malicious) peers. These defences are cheap:

- **Application-level checksums** on payloads (TCP's checksum is weak; Ethernet's is overridden when a switch rewrites a frame).
- **Input validation and sanitisation** at every service boundary, especially for public-facing APIs.
- **Multi-source NTP**: compare several servers and drop outliers, which is exactly how NTP clients defend against a single misconfigured time server.

```mermaid
graph LR
  subgraph BFT_needed[Full BFT is worth paying for]
    A[Aerospace / defence]
    B[Public blockchains]
    P[Adversarial P2P]
  end
  subgraph Weak_defences[Cheap weak defences suffice]
    D[Shared-tenant datacenter]
    E[Single-organisation cluster]
    F[Internet-facing API]
  end
```

See [[05-byzantine-faults]].

---

## Deep Dive 6 — System models: safety and liveness

To reason about an algorithm you must fix a **system model** — a pair of assumptions:

**Timing model** (how the network and clocks behave):

```mermaid
graph LR
  Sync[Synchronous<br/>bounded delay + pauses + clock error] --> PSync[Partially synchronous<br/>bounded most of the time<br/>occasionally unbounded]
  PSync --> Async[Asynchronous<br/>no timing assumptions at all]
```

**Node-failure model** (how nodes misbehave):

```mermaid
graph LR
  CS[Crash-stop<br/>node dies, never returns] --> CR[Crash-recovery<br/>node can reboot with stable storage]
  CR --> LS[Limping / fail-slow<br/>node alive but 100x slower]
  LS --> BY[Byzantine<br/>node lies arbitrarily]
```

The model most real systems target is **partially synchronous + crash-recovery** — it admits real-world pauses and reboots while still allowing progress during stable periods. Within a chosen model, correctness is split into two kinds of property:

- **Safety**: "nothing bad ever happens" (no two leaders, no lost committed write, no double spend). Safety violations are **permanent and visible** — you can point to the moment the invariant broke. Safety must hold *always*, even while the network is broken.
- **Liveness**: "something good eventually happens" (every request eventually gets a response, every leader is eventually elected). Liveness violations are **conditional** — they usually come with a caveat ("*provided* a majority is reachable") and are satisfied by waiting longer, not by proving a point-in-time fact.

Algorithms are designed so that **safety never depends on timing** (it holds even in the worst asynchronous model) while **liveness is conditional on partial synchrony**. This is why Raft and Paxos stay safe during a partition and only make progress once the partition heals.

See [[06-system-models-safety-and-liveness]].

---

## Deep Dive 7 — Formal methods and deterministic testing

The number of ways a distributed system can fail is astronomical — a handful of nodes with crash, pause, message-drop, and message-reorder faults already has a state space beyond what conventional testing can explore. Modern distributed-systems engineering relies on a **layered toolkit**:

```mermaid
graph TD
  Spec[TLA+ / FizzBee<br/>spec-level model checking] --> Code[Real implementation]
  Code --> FI[Jepsen fault injection<br/>kill -9, iptables DROP,<br/>clock skew, slow disk]
  Code --> DST[Deterministic Simulation<br/>mock clock, mock network,<br/>single-threaded scheduler]
  FI --> Report[Bug report]
  DST --> Replay[Exact seed -> exact replay]
```

- **Model checking (TLA+, FizzBee, Alloy, P)**: you write the algorithm at an abstract level and the checker explores every reachable state against safety/liveness invariants. Fast, catches subtle protocol bugs, but the spec can drift from the real code.
- **Fault injection (Jepsen, Chaos Mesh, Gremlin)**: exercise the real binary against induced crashes, partitions, clock skew, slow disks, and check for invariant violations. Catches implementation bugs but is **hard to reproduce** — the faulty interleaving rarely recurs on rerun.
- **Deterministic simulation testing (DST)**: run the real code through a single-threaded scheduler with mocked clocks, networks, and I/O. Every non-determinism (time, random numbers, thread scheduling) is seeded, so a failing run can be **replayed exactly**. Pioneered by **FoundationDB** (application-level, written in their Flow language), used by **TigerBeetle** (custom event loop), and offered as a service by **Antithesis** (hypervisor-level, any binary).

DST is becoming the standard for new storage systems because "I hit a bug I cannot reproduce" is no longer an acceptable answer.

See [[07-formal-methods-and-deterministic-testing]].

---

## Trade-offs: timing models

| Model | Bounded delay? | Bounded pauses? | Bounded clock error? | Realistic for? |
|---|---|---|---|---|
| **Synchronous** | yes | yes | yes | almost nothing — useful only as a theoretical baseline. |
| **Partially synchronous** | usually, occasionally no | usually, occasionally no | usually, occasionally no | most real datacenter systems; target of Raft/Paxos/ZAB. |
| **Asynchronous** | no | no | no | the worst case you prove safety against; no algorithm can guarantee progress here (FLP). |

## Trade-offs: node failure models

| Model | Assumption | Tolerates `f` failures in cluster of | Cost | Used by |
|---|---|---|---|---|
| **Crash-stop** | crashed nodes never come back | `2f+1` | cheapest | simple research protocols; naive Raft in memory. |
| **Crash-recovery** | crashed nodes may reboot with durable storage | `2f+1` | moderate (stable-storage writes) | Raft, Paxos, ZooKeeper — the real-world default. |
| **Limping / fail-slow** | node alive but orders of magnitude slower | no clean bound; needs application-level health signals | ops pain | seen in practice; handled by hedged requests and greylist eviction. |
| **Byzantine** | node lies arbitrarily | `3f+1` | highest — multi-implementation, signed messages | PBFT, blockchain consensus, aerospace triple-modular redundancy. |

---

## Key Takeaways

1. **Partial failure is the defining feature** of a distributed system — assume every message may be lost, every reply may be delayed, every node may be paused, and design accordingly.
2. **A timeout is always a guess.** The six indistinguishable outcomes (lost request, queued request, crashed node, paused node, lost reply, delayed reply) are a fact of asynchronous networks, not a bug you can fix.
3. **Clocks are confidence intervals, not points.** Wall-clock timestamps for ordering are unsafe unless you expose the interval (TrueTime, ClockBound) and wait it out, or use logical clocks.
4. **Any thread can be paused for seconds.** A valid lease at line *n* may be expired at line *n+1*. The only durable defence is **fencing tokens checked by the storage layer**.
5. **Majority quorums and fencing tokens are the foundation** of nearly every correct distributed algorithm — they survive zombies, partitions, and delayed packets alike.
6. **Pick your failure model honestly.** Most production systems are **partially synchronous + crash-recovery**; BFT is overkill outside of aerospace, blockchains, and adversarial P2P, but cheap Byzantine-lite defences (checksums, input validation, multi-source NTP) pay off everywhere.
7. **Correctness beyond code review requires model checking + fault injection + deterministic simulation testing.** The state space is too large for hope; modern databases that take durability seriously adopt all three.

---

## See Also

- [[01-unreliable-networks-and-fault-detection]]
- [[02-unreliable-clocks]]
- [[03-process-pauses]]
- [[04-quorums-leases-and-fencing-tokens]]
- [[05-byzantine-faults]]
- [[06-system-models-safety-and-liveness]]
- [[07-formal-methods-and-deterministic-testing]]